In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
files = {
    "caract": "caract-2023.csv",
    "lieux": "lieux-2023.csv",
    "usagers": "usagers-2023.csv",
    "vehicules": "vehicules-2023.csv",
}

def load_raw(path):
    """Load a BAAC CSV as text and normalise whitespace and missing markers."""
    df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    # Strip regular, non-breaking and narrow non-breaking spaces at both ends of each cell
    df = df.apply(lambda s: s.str.strip().str.strip("\u00a0\u202f"))
    return df.replace({"": np.nan, "N/A": np.nan})

dfs = {k: load_raw(RAW / f) for k, f in files.items()}

# Harmonise the accident key name across years (some files use "Accident_Id")
for df in dfs.values():
    df.rename(columns={"Accident_Id": "Num_Acc"}, inplace=True)

# Clean IDs: remove all inner spaces (regular, non-breaking, narrow)
for df in dfs.values():
    for col in ["id_usager", "id_vehicule"]:
        if col in df.columns:
            df[col] = df[col].str.replace("[\\s\u00a0\u202f]", "", regex=True)

# Coordinates: comma decimal -> float
for col in ["lat", "long"]:
    dfs["caract"][col] = pd.to_numeric(
        dfs["caract"][col].str.replace(",", ".", regex=False), errors="coerce"
    )

for k, df in dfs.items():
    print(f"--- {k}: {df.shape}")
    print(list(df.columns))

dfs["caract"][["lat", "long"]].describe()

--- caract: (54822, 15)
['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long']
--- lieux: (70860, 18)
['Num_Acc', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma']
--- usagers: (125789, 16)
['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'place', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp']
--- vehicules: (93585, 11)
['Num_Acc', 'id_vehicule', 'num_veh', 'senc', 'catv', 'obs', 'obsm', 'choc', 'manv', 'motor', 'occutc']


,lat,long
count,54822.000000,54822.000000
mean,44.250545,1.830744
std,12.419628,18.959204
min,-23.372504,-176.207000
25%,44.790693,1.172116
50%,47.859229,2.384819
75%,48.861237,4.734031
max,51.047490,168.095670


In [22]:
for k, df in dfs.items():
    miss = pd.DataFrame({
        "nan_%": df.isna().mean() * 100,
        "moins1_%": (df.astype(str).apply(lambda s: s.str.strip()) == "-1").mean() * 100,
    }).round(1)
    print(f"--- {k}")
    print(miss[(miss > 0).any(axis=1)])

--- caract
     nan_%  moins1_%
adr    2.5       0.0
--- lieux
         nan_%  moins1_%
voie      18.0       0.0
v1         0.0      23.2
v2        91.7       0.0
circ       0.0       7.1
nbv        0.0       5.9
vosp       0.0       5.6
prof       0.0       0.3
pr         0.0      42.0
pr1        0.0      42.3
plan       0.0       0.2
lartpc   100.0       0.0
larrout    0.0      74.2
surf       0.0       0.3
infra      0.0       1.3
situ       0.0       0.3
vma        0.0       6.5
--- usagers
         nan_%  moins1_%
grav       0.0       0.1
sexe       0.0       1.9
an_nais    2.1       0.0
trajet     0.0       2.0
secu1      0.0       1.8
secu2      0.0      44.1
secu3      0.0      98.1
locp       0.0      50.3
actp       0.0      50.3
etatp      0.0      92.0
--- vehicules
        nan_%  moins1_%
senc      0.0       0.1
obs       0.0       0.1
choc      0.0       0.1
motor     0.0       0.2
occutc   99.1       0.0


### Missing values (2023)

Missing data appears in two forms: true empty values (`NaN`) and the code `-1` ("not specified").

- **`caract`**: almost complete; only `adr` (free-text address, 2.5%) has gaps. Coordinates are complete.
- **Unusable columns (dropped)**: `lartpc` (100% empty), `occutc` (99.1%, only relevant for public transport),
  `v2` (91.7%), `larrout` (74.2%), `pr`/`pr1` (~42%), `v1` (23.2%).
- **Kept with caution**: `vma` (speed limit, 6.5% unknown), `circ`, `nbv`, `vosp` (6–7%).
  Unknown values are shown as an "Unknown" category, not imputed.
- **Key variables are nearly complete**: `grav` (0.1% unknown, excluded from severity metrics), `sexe` (1.9%),
  `obs`, `choc`, `motor` (≤ 0.2%).
- **`usagers` structural gaps**: `locp`, `actp` (50.3%) and `etatp` (92.0%) only apply to pedestrians;
  `secu2` (44.1%) and `secu3` (98.1%) are optional extra safety equipment → not used.
- **`an_nais`** (year of birth, used to compute age): 2.1% missing → age shown as "Unknown".

In [23]:
key = "Num_Acc"
ids = set(dfs["caract"][key])
for k in ["lieux", "vehicules", "usagers"]:
    orphans = (~dfs[k][key].isin(ids)).sum()
    print(k, "orphelins:", orphans, "| doublons de clé:", dfs[k][key].duplicated().sum())

lieux orphelins: 0 | doublons de clé: 16038
vehicules orphelins: 0 | doublons de clé: 38763
usagers orphelins: 0 | doublons de clé: 70967


### Join integrity check (2023)

**Orphans**: 0 in `lieux`, `vehicules` and `usagers` → every row links to an accident in `caract`. 

**`Num_Acc` duplicates**: to be interpreted according to each table's 
| Table | 1 row = | `Num_Acc` duplicates | Verdict |
|---|---|---|---|
| `vehicules` | 1 vehicle | 38,763 | Expected: several vehicles per accident |
| `usagers` | 1 person | 70,967 | Expected: several people per accident |
| `lieux` | 1 accident (in principle) | 16,038 |  Anomaly to investigate |

**Consequence**: a direct `caract` + `lieux` join would duplicate some accidents and inflate totals.


In [ ]:
# Uniqueness at the correct level of granularity
checks = {
    "vehicules": ["Num_Acc", "id_vehicule"],
    "usagers": ["Num_Acc", "id_usager"] if "id_usager" in dfs["usagers"].columns
               else ["Num_Acc", "id_vehicule", "place", "an_nais"],
}
for k, cols in checks.items():
    print(k, cols, "-> duplicates:", dfs[k].duplicated(subset=cols).sum())


for k, df in dfs.items():
    print(k, "identical rows:", df.duplicated().sum())

vehicules ['Num_Acc', 'id_vehicule'] -> duplicates: 0
usagers ['Num_Acc', 'id_usager'] -> duplicates: 0
caract identical rows: 0
lieux identical rows: 0
usagers identical rows: 0
vehicules identical rows: 0


In [25]:
lieux = dfs["lieux"]
dup = lieux[lieux["Num_Acc"].duplicated(keep=False)].sort_values("Num_Acc")
print(dup["Num_Acc"].nunique(), "accidents with several location rows")
print(dup["Num_Acc"].value_counts().value_counts().sort_index())  # rows per accident

# Which columns differ between rows of the same accident?
diff_cols = (dup.groupby("Num_Acc").nunique() > 1).mean().sort_values(ascending=False)
print(diff_cols[diff_cols > 0].round(3))
dup.head(10)

15870 accidents with several location rows
count
2    15715
3      143
4       11
5        1
Name: count, dtype: int64
nbv        0.540
voie       0.525
vosp       0.413
circ       0.379
pr1        0.340
pr         0.329
vma        0.326
v1         0.305
prof       0.177
plan       0.149
infra      0.148
larrout    0.129
situ       0.086
surf       0.022
v2         0.004
dtype: float64


,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202300000001,4,RUE DE RIVOLI,0,NaN,1,2,0,1,-1,-1,1,NaN,-1,2,0,1,30
1,202300000001,4,RUE SAINT FLORENTIN,0,NaN,1,1,0,1,-1,-1,1,NaN,-1,2,0,1,30
3,202300000003,3,5,0,NaN,2,4,0,1,1,0,1,NaN,-1,2,5,1,50
4,202300000003,3,87,0,NaN,2,4,0,1,1,0,1,NaN,-1,2,5,1,50
6,202300000005,4,NaN,0,NaN,1,1,0,1,-1,-1,1,NaN,-1,2,0,1,30
7,202300000005,4,NaN,0,NaN,2,2,0,1,-1,-1,1,NaN,-1,2,0,1,30
10,202300000008,3,130,0,NaN,3,4,0,1,-1,-1,1,NaN,-1,2,0,1,50
11,202300000008,3,NaN,0,NaN,2,2,0,1,-1,-1,1,NaN,-1,2,0,1,50
12,202300000009,3,5,0,NaN,3,4,0,1,-1,-1,1,NaN,-1,2,5,1,50
13,202300000009,3,87,0,NaN,1,2,0,1,-1,-1,1,NaN,-1,1,5,1,50


In [ ]:
# Are multi-location accidents mostly at intersections? (int == "1" means "no intersection")
caract = dfs["caract"]
multi = caract["Num_Acc"].isin(dup["Num_Acc"])
print("At an intersection - multi-location: ", round((caract.loc[multi, "int"] != "1").mean() * 100, 1), "%")
print("At an intersection - single-location:", round((caract.loc[~multi, "int"] != "1").mean() * 100, 1), "%")

# Rule: keep the main road 
lieux_dedup = (lieux.assign(catr_num=pd.to_numeric(lieux["catr"], errors="coerce"))
                    .sort_values(["Num_Acc", "catr_num"])
                    .drop_duplicates(subset="Num_Acc", keep="first")
                    .drop(columns="catr_num"))
print(len(lieux), "->", len(lieux_dedup), "rows |", len(lieux) - len(lieux_dedup), "removed")
print("All accidents preserved:", lieux_dedup["Num_Acc"].nunique() == caract["Num_Acc"].nunique())

At an intersection - multi-location:  100.0 %
At an intersection - single-location: 10.6 %
70860 -> 54822 rows | 16038 removed
All accidents preserved: True


### Multiple location rows per accident (`lieux`)

15,870 accidents have more than one row in `lieux` (15,715 with 2 rows, a few with up to 5).
No row is an exact duplicate: the differing columns (`voie`, `nbv`, `vosp`, `circ`, `pr1`) describe
**different roads involved in the same accident**, most likely at intersections
.

**Rule applied**: keep one row per accident, the main road (lowest `catr` code = highest road category).
Result: XX rows removed, every accident preserved. This prevents double-counting when joining `caract` and `lieux`.

In [ ]:
#  number of people killed in 2023 (grav == "2" = killed)
u = dfs["usagers"].merge(dfs["caract"][["Num_Acc", "dep"]], on="Num_Acc")
killed = (u["grav"] == "2")
overseas = u["dep"].str.len() == 3   # 971, 972, 974, 976, 986, 987, 988...
print("Accidents:", len(dfs["caract"]))
print("People involved:", len(u))
print("Killed - all territories:", killed.sum())
print("Killed - metropolitan France:", (killed & ~overseas).sum())

Accidents: 54822
People involved: 125789
Killed - all territories: 3398
Killed - metropolitan France: 3167


### Validation & geography

- **External validation — people killed in 2023**: 3,398 in the data (all territories) and 3,167 (metropolitan France),
  **exactly matching** the official ONISR final figures (3,398 and 3,167).
  → Confirms the tables, the joins and the `grav` coding are correctly understood.
- **Known bias (ONISR)**: police are not systematically called to non-fatal accidents, so injury accidents
  and slight injuries are under-reported. Fatalities are complete → the most reliable indicator for comparisons.
- **Coordinates**: complete and numeric, but they span all French territories
  (longitude from -176 to 168: overseas territories such as Wallis-and-Futuna and New Caledonia).
  → The map will focus on **metropolitan France** (overseas shown separately or filtered out).

Source: ONISR, *Bilan 2023 de la sécurité routière* (final results).